In [0]:
# 03b — Gold: Demanda estimada vs. cobertura por barrio

import pandas as pd
from datetime import datetime, timezone

In [0]:
CATALOG = "gobierno_abierto"
SCHEMA_SILVER = "b_silver"
SCHEMA_GOLD = "c_gold"

In [0]:
## 1. Lectura de insumos

route_stops = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.route_stops").toPandas()
promedio_empresa = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.cortes_boleto_promedio_empresa").toPandas()
stops_by_barrio = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.stops_by_barrio").toPandas()
barrio_coverage = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.barrio_coverage").toPandas()

# Solo agencias con dato de demanda Y con al menos una parada mapeada
route_stops = route_stops.dropna(subset=["agency_name", "stop_id"])
print(f"route_stops (con agencia): {len(route_stops)}")
print(f"Empresas con promedio diario: {list(promedio_empresa['empresa'])}")
print(f"Empresas con paradas mapeadas en GTFS: {sorted(route_stops['agency_name'].unique())}")

In [0]:
## 2. Paradas por empresa y por barrio

stop_agencia = route_stops[["stop_id", "agency_name"]].drop_duplicates()

paradas_por_empresa_barrio = (
    stop_agencia.merge(stops_by_barrio[["stop_id", "barrio_id"]], on="stop_id", how="inner")
    .dropna(subset=["barrio_id"])
    .groupby(["agency_name", "barrio_id"])
    .size()
    .rename("cantidad_paradas_empresa")
    .reset_index()
)

print(f"Combinaciones empresa-barrio: {len(paradas_por_empresa_barrio)}")
paradas_por_empresa_barrio.head(10)

In [0]:
## 3. Reparto proporcional del promedio diario por barrio

promedio_empresa["empresa_norm"] = promedio_empresa["empresa"].str.strip().str.upper()
paradas_por_empresa_barrio["agency_norm"] = paradas_por_empresa_barrio["agency_name"].str.strip().str.upper()

sin_match = set(paradas_por_empresa_barrio["agency_norm"]) - set(promedio_empresa["empresa_norm"])
if sin_match:
    print(f"⚠️ Empresas en GTFS sin promedio de demanda (revisar normalización de nombres): {sin_match}")

# COMMAND ----------

total_paradas_por_empresa = (
    paradas_por_empresa_barrio.groupby("agency_norm")["cantidad_paradas_empresa"]
    .sum()
    .rename("total_paradas_empresa")
    .reset_index()
)

reparto = paradas_por_empresa_barrio.merge(total_paradas_por_empresa, on="agency_norm", how="left")
reparto = reparto.merge(
    promedio_empresa[["empresa_norm", "promedio_diario"]], left_on="agency_norm", right_on="empresa_norm", how="inner"
)

reparto["demanda_estimada_empresa_barrio"] = (
    reparto["promedio_diario"] * reparto["cantidad_paradas_empresa"] / reparto["total_paradas_empresa"]
)

demanda_por_barrio = (
    reparto.groupby("barrio_id")["demanda_estimada_empresa_barrio"]
    .sum()
    .rename("demanda_estimada")
    .reset_index()
)

print(f"Barrios con demanda estimada: {len(demanda_por_barrio)}")
demanda_por_barrio.sort_values("demanda_estimada", ascending=False).head(10)

In [0]:
## 4. `gold.demanda_vs_cobertura`

demanda_vs_cobertura = barrio_coverage[
    ["barrio_id", "barrio_nombre", "area_km2", "cantidad_paradas", "paradas_por_km2", "ranking_cobertura"]
].merge(demanda_por_barrio, on="barrio_id", how="left")

demanda_vs_cobertura["ranking_demanda"] = (
    demanda_vs_cobertura["demanda_estimada"].rank(method="min", ascending=False).astype("Int64")
)

n = demanda_vs_cobertura["barrio_id"].nunique()
demanda_vs_cobertura["percentil_cobertura"] = 1 - (demanda_vs_cobertura["ranking_cobertura"] - 1) / n
demanda_vs_cobertura["percentil_demanda"] = 1 - (demanda_vs_cobertura["ranking_demanda"] - 1) / n
demanda_vs_cobertura["desajuste_demanda_cobertura"] = (
    demanda_vs_cobertura["percentil_demanda"] - demanda_vs_cobertura["percentil_cobertura"]
)

demanda_vs_cobertura = demanda_vs_cobertura.sort_values("desajuste_demanda_cobertura", ascending=False)
demanda_vs_cobertura["_created_at"] = datetime.now(timezone.utc)

sin_demanda = demanda_vs_cobertura["demanda_estimada"].isna().sum()
print(f"⚠️ Barrios sin ninguna parada de Coniferal/ERSA/TAMSE (sin demanda estimable): {sin_demanda}")

demanda_vs_cobertura.head(10)

In [0]:
spark.createDataFrame(demanda_vs_cobertura).write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.demanda_vs_cobertura")

count = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.demanda_vs_cobertura").count()
print(f"{CATALOG}.{SCHEMA_GOLD}.demanda_vs_cobertura: {count} filas escritas.")

In [0]:
## Verificación final — top desajustes en ambos sentidos

print("Barrios con MÁS demanda relativa que cobertura (candidatos a reforzar paradas):")
display(
    spark.table(f"{CATALOG}.{SCHEMA_GOLD}.demanda_vs_cobertura")
    .orderBy("desajuste_demanda_cobertura", ascending=False)
    .limit(10)
)

print("\nBarrios con MÁS cobertura relativa que demanda (posible sobre-oferta):")
display(
    spark.table(f"{CATALOG}.{SCHEMA_GOLD}.demanda_vs_cobertura")
    .orderBy("desajuste_demanda_cobertura", ascending=True)
    .limit(10)
)